# *Klebsiella pneumoniae* AMR: Metadata Cleaning, Grouping and Sampling Workflow

**Goal.** Starting from the NCBI Pathogen Detection metadata sheet for *K. pneumoniae* (`Kp_all_metadata.csv`), I build

1. a cleaned metadata table with two new categorical columns, `Isolation_Group` and `Continent`, and
2. a balanced, stratified random sample of **300 genomes** (3 isolation groups × 5 continents × 20 genomes), together with one accession list per isolation group that I can use to download the assemblies for the AMR analysis.

| File | Role |
|---|---|
| `Kp_all_metadata.csv` | input: raw NCBI Pathogen Detection metadata |
| `kp_isolation_groups.py` | module used in step 8 (builds `Isolation_Group`) |
| `kp_continent_groups.py` | module used in step 10 (builds `Continent`) |
| `Kp_clean.csv` | output of Part A: cleaned table |
| `Kp_sample.csv` | output of Part B: the 300 sampled genomes |
| `clinical_assembly.txt`, `environmental_assembly.txt`, `animal_assembly.txt` | output of Part B: one accession per line |

**How to run:** put the CSV and the two `.py` modules in the same folder as this notebook, install `pandas` (`pip install pandas`), then run the cells from top to bottom.

## Workflow overview

**Part A: cleaning the metadata**

1. Load the raw CSV.
2. Drop unused columns (`Strain`, `Isolate`, `BioSample`) and rows with real empty cells.
3. Pull a 4-digit collection year out of the messy `Collection_date` text.
4. Lowercase and strip the text, and convert hidden missing words ("not collected", "unknown", ...) to real `NaN`.
5. Set impossible years (1900 placeholders, anything outside 1990–2025) to missing.
6. Drop rows missing `Location`, `Isolation_source` or `Collection_year`.
7. Audit: confirm zero `NaN` and zero leftover missing words.
8. Create the `Isolation_Group` column from `Isolation_source` (Clinical / Environmental / Animal / Unclassified).
9. Save `Kp_clean.csv`.

**Part B: sampling**

10. Create the `Continent` column from `Location`.
11. Keep genomes collected in 2018 or later.
12. Count genomes in every isolation group × continent box to make sure each has enough.
13. Randomly pick 20 genomes per box (3 groups × 5 continents = 300) with a fixed seed so the result is reproducible.
14. Save `Kp_sample.csv` and one accession list per isolation group.

## Setup and settings

Every tunable parameter is a module-level constant in this one cell, so I never have to edit the function bodies further down. What the constants do:

- `COLS_TO_DROP`: identifier columns that carry no information for grouping or sampling.
- `REQUIRED_COLS`: the three variables every kept row must have (they are needed for the year filter, the isolation group and the continent). This is a complete-case filter.
- `YEAR_MIN` / `YEAR_MAX`: window of plausible collection years. The year 1900 appears 40 times in the sheet and is clearly a placeholder, not a real date. (The dataset also has 2026 genomes, see the notes at the end.)
- `MISSING_TOKENS`: all the different ways people wrote "I don't know" in the NCBI sheet. It is a `set`, so membership tests with `isin` are fast.
- `ISOLATION_MISSING_TOKENS`: extra junk strings I found by searching `Isolation_source` by hand. They are exact strings and are only applied to that column. (`"not detemined"` is a typo that is really in the original data, so it is kept on purpose.)
- `SAMPLE_MIN_YEAR`: only genomes from this year onward go into the sample (newer sequencing data).
- `SAMPLE_GROUPS` × `SAMPLE_CONTINENTS`: the strata ("boxes") for stratified sampling. Oceania is left out of `SAMPLE_CONTINENTS` on purpose: it has 0 animal genomes from 2018 onward, so I cannot draw 20 from every box. It is still shown in the counts table through `ALL_CONTINENTS`.
- `N_PER_BOX` and `RANDOM_SEED`: genomes drawn per box, and the seed that makes the draw reproducible.
- `DROP_GCA_DUPLICATES`: `False` reproduces my original notebook. `True` also removes the GenBank (`GCA_`) copy when the same genome is present as RefSeq (`GCF_`), see note 2 at the end.

In [ ]:
import pandas as pd

In [ ]:
RAW_FILE = "Kp_all_metadata.csv"
CLEAN_FILE = "Kp_clean.csv"
SAMPLE_FILE = "Kp_sample.csv"

# ---- part A settings ----
# columns that are not useful for this analysis
COLS_TO_DROP = ["Strain", "Isolate", "BioSample"]

# columns that must be filled in for a row to be kept
REQUIRED_COLS = ["Location", "Isolation_source", "Collection_year"]

# years outside this window are treated as errors
# (1900 shows up 40 times, it is clearly a placeholder, not a real date)
YEAR_MIN = 1990
YEAR_MAX = 2025   # NOTE: the dataset also has 2026 genomes, see the notes at the bottom

# all the different ways people wrote "i don't know" in the NCBI sheet
MISSING_TOKENS = {
    "", "-", "--", ".", "?", "n/a", "n.a", "n.a.", "na", "nan", "nil", "none", "null",
    "null value", "nd", "n.d.", "#n/a", "#na", "tbd", "pending", "unk",
    "missing", "unknown", "unspecified", "undetermined", "unavailable",
    "no data", "no information",
    "not applicable", "not available", "not collected", "not determined",
    "not found", "not known", "not provided", "not reported",
    "restricted access",
}

# extra junk i found by searching Isolation_source by hand
# (these are all exact strings, i checked them one by one)
ISOLATION_MISSING_TOKENS = {
    "other (unknown)", "other/unknown", "unknown/unspecified", "unknown isolate",
    "unknown material", "not assigned", "not detemined",   # (typo is in the original data)
    "not available: to be reported later",
    "unspecified diagnostic sample", "urine unspecified", "sputum unspecified",
    "wound unspecified", "urine,type not specified",
    "human, missing", "dog, not reported",
}

# ---- part B settings ----
# only genomes from this year onward go into the sample (newer sequencing data)
SAMPLE_MIN_YEAR = 2018

# the boxes for stratified sampling: isolation group x continent
SAMPLE_GROUPS = ["Clinical", "Environmental", "Animal"]

# Oceania is NOT in this list on purpose: it has 0 animal genomes after 2018,
# so i can't pull 20 from every box. (it is still shown in the counts table)
SAMPLE_CONTINENTS = ["Asia", "Europe", "North America", "Africa", "South America"]
ALL_CONTINENTS = SAMPLE_CONTINENTS + ["Oceania"]

N_PER_BOX = 20       # genomes picked from each group x continent box
RANDOM_SEED = 42     # same seed = same 300 genomes every time i run it

# False = same as my notebook. True = also remove the GCA_ copy when the same
# genome is there as GCF_ too (see note 2 at the bottom)
DROP_GCA_DUPLICATES = False

### Helper: `log_rows`

After every filtering step I print the number of rows left, so I can see how many rows each step removes. In the f-string, `{message:<55}` left-aligns the text in 55 characters and `{len(df):>9,}` right-aligns the row count with a thousands separator.

In [ ]:
def log_rows(df, message):
    print(f"{message:<55} {len(df):>9,} rows")

# Part A: Cleaning the metadata

### Steps 1–2: load the file and drop what I don't need

`pd.read_csv` loads the raw sheet. `drop(columns=COLS_TO_DROP)` removes the identifier columns, and `dropna()` then removes every row that has a real empty cell (`NaN`) in any remaining column. This only catches *real* missing values. Text such as `"missing"` is just a string to pandas, so it survives here and is handled in step 4.

In [ ]:
def load_metadata(path):
    df = pd.read_csv(path)
    log_rows(df, "loaded raw file")
    return df

In [ ]:
def drop_unused_columns(df):
    df = df.drop(columns=COLS_TO_DROP)
    # rows with a totally empty cell (real NaN) can't be used either
    df = df.dropna()
    log_rows(df, "after dropping columns + real NaN rows")
    return df

In [ ]:
df = load_metadata(RAW_FILE)
df.head()

In [ ]:
df = drop_unused_columns(df)

### Step 3: extract the collection year

`Collection_date` comes in many formats (`"2010"`, `"2012-05"`, `"2012-02-28"`, `"2018/2019"`, `"missing"`, `"not collected"`, ...). Instead of parsing dates, I take the **first run of 4 digits** with the regular expression `(\d{4})`. `str.extract` returns the capture group, and rows without any 4-digit number get `NaN`.

- `astype(str)` first, so non-string values cannot break the `.str` accessor.
- `pd.to_numeric(..., errors="coerce")` turns the extracted text into numbers (anything unparseable becomes `NaN`).
- `astype("Int64")` is pandas' *nullable* integer type: it can hold missing values without converting the whole column to float (so years stay `2012`, not `2012.0`).

Then the original `Collection_date` column is dropped and rows without a year are removed.

In [ ]:
def extract_year(df):
    """
    Collection_date comes in lots of formats: "2010", "2012-05", "2012-02-28",
    "2018/2019", "missing", "not collected"... so i just grab the first
    4-digit number and call that the year. anything without one becomes NaN.
    """
    year = df["Collection_date"].astype(str).str.extract(r"(\d{4})")[0]
    df["Collection_year"] = pd.to_numeric(year, errors="coerce").astype("Int64")

    df = df.drop(columns="Collection_date")
    df = df.dropna(subset=["Collection_year"])
    log_rows(df, "after extracting year (no-year rows removed)")
    return df

In [ ]:
df = extract_year(df)

### Step 4: tidy the text and turn hidden missing values into real `NaN`

For every text column I normalise the strings and then replace the "missing-type" words with real `NaN`:

1. `astype("string")` converts to pandas' `StringDtype`, which stores missing values as `pd.NA`.
2. `.str.strip().str.lower()` removes surrounding spaces and lowercases, so `"Unknown "` and `"UNKNOWN"` both become `"unknown"`. Lowercasing has to happen *before* the comparison because the token sets are lowercase.
3. `s.isin(tokens)` gives a boolean mask and `s.mask(...)` replaces the `True` positions with `NaN`.

For `Isolation_source` the token set is the union (`|`) of the general tokens and the extra isolation-specific ones. Side effect: *every* text column is now lowercase, including `Assembly` (see note 3 at the end).

In [ ]:
def clean_text_columns(df):
    """
    lowercase + strip spaces on every text column, then replace the
    "missing"-type words with real NaN so pandas can see them.
    (before this, dropna() didn't catch them because 'missing' is just a string)
    """
    text_cols = df.select_dtypes(include=["object", "string"]).columns

    for col in text_cols:
        s = df[col].astype("string").str.strip().str.lower()

        tokens = MISSING_TOKENS
        if col == "Isolation_source":
            tokens = MISSING_TOKENS | ISOLATION_MISSING_TOKENS

        df[col] = s.mask(s.isin(tokens))   # True -> becomes NaN

    return df

In [ ]:
df = clean_text_columns(df)

### Step 5: remove impossible years

`between(YEAR_MIN, YEAR_MAX)` is a boolean mask that is `True` for plausible years (both ends inclusive), and `~` inverts it so `bad` marks the impossible ones. I print how many there are and set them to `pd.NA` with `.loc`. They are not deleted yet: step 6 drops them together with every other incomplete row.

In [ ]:
def remove_bad_years(df):
    bad = ~df["Collection_year"].between(YEAR_MIN, YEAR_MAX)
    print(f"years outside {YEAR_MIN}-{YEAR_MAX}: {bad.sum():,}")
    df.loc[bad, "Collection_year"] = pd.NA
    return df

In [ ]:
df = remove_bad_years(df)

### Step 6: drop rows that are still incomplete

`dropna(subset=REQUIRED_COLS)` only looks at the three columns I need downstream (complete-case filtering). `reset_index(drop=True)` renumbers the rows from 0 so the index has no gaps.

In [ ]:
def drop_incomplete_rows(df):
    df = df.dropna(subset=REQUIRED_COLS).reset_index(drop=True)
    log_rows(df, "after dropping incomplete rows")
    return df

In [ ]:
df = drop_incomplete_rows(df)

### Step 7: audit, verify instead of assume

A final check that the cleaning really worked. For each column I count (a) real `NaN` values and (b) leftover missing words that are still sitting in the text as strings. Both counts must be 0 everywhere. The function prints the report table and a pass/fail message, and also returns the report as a DataFrame.

In [ ]:
def audit_missing(df):
    """
    counts real NaN and also any leftover 'missing' words, per column.
    both should be 0 at the end.
    """
    report = []
    for col in df.columns:
        nan_count = int(df[col].isna().sum())

        leftover = 0
        if df[col].dtype == "object" or str(df[col].dtype) == "string":
            text = df[col].astype("string").str.strip().str.lower()
            leftover = int(text.isin(MISSING_TOKENS | ISOLATION_MISSING_TOKENS).sum())

        report.append({"column": col, "nan": nan_count, "hidden_missing_words": leftover})

    report = pd.DataFrame(report)
    print(report.to_string(index=False))

    if report[["nan", "hidden_missing_words"]].sum().sum() == 0:
        print("clean, nothing missing left")
    else:
        print("still dirty, check the columns above")
    return report

In [ ]:
audit_report = audit_missing(df)

### Step 8: create `Isolation_Group` from `Isolation_source` (Module 1)

The sheet has **2,908 different `Isolation_source` strings** (`"blood"`, `"cow milk"`, `"hospital effluent"`, plus many typos and alternative spellings), far too many to analyse one by one. I collapse them into a categorical variable with four levels, following the One Health idea (human / animal / environment):

| Group | What goes in |
|---|---|
| **Clinical** | human specimens and body sites (blood, urine, sputum, wound, stool, CSF, catheters, carriage swabs, named hospitals/wards used as the specimen source) |
| **Environmental** | non-host matrices: water, wastewater, sludge, soil, air, manure, plants and plant food, farm premises, and hospital surfaces (sinks, drains, beds, rails, ...) |
| **Animal** | non-human hosts (livestock, poultry, pets, wildlife, insects, fish), their body sites and faeces, and food of animal origin (meat, milk, eggs, seafood) |
| **Unclassified** | not real isolation sources: lab-derived strains (mutants, evolved lines, ATCC passages, phage selection), bare codes, city names, "other", "(blank)". They are kept separate on purpose, not folded into the other three |

The function `assign_group()` lives in its own file, `kp_isolation_groups.py` (~3,470 lines, mostly the lists), and works in **two layers**:

1. **Curated lists (checked first).** Four hand-checked lists: Clinical 2,029 strings, Environmental 431, Animal 307, Unclassified 141. They are turned into a dictionary `SOURCE_TO_GROUP` (lowercased and also *normalised* by `norm()`, which strips punctuation and collapses whitespace), so a lookup is a single dict access.
2. **Rule engine (fallback).** If the string is not in the lists, `classify()` applies keyword rules (regular expressions) so new or misspelled sources from future downloads still get a group. The first rule that matches wins:
   1. exact overrides for ambiguous words (107 hand-resolved cases, e.g. `"hospital sink"` → Environmental);
   2. explicit "environment/environmental" wording → Environmental;
   3. surface/fixture words (sink, floor, bed rail, ...) without a real specimen word → Environmental;
   4. institution words (hospital, clinic, ...) without environmental words → Clinical;
   5. lab-derived words (mutant, in vitro, ATCC, phage, ...) → Unclassified;
   6. an explicit human mention → Clinical (it wins over an animal word, e.g. "farm worker");
   7. an animal host word → Animal, unless a matrix/premises word is present (manure, water, farm, ...) → Environmental;
   8. then, in this order: animal-origin food without a named species (follows the `ANIMAL_DERIVED_FOOD_AS` switch), plant/produce → Environmental, environmental matrices → Environmental (but Clinical if a real body specimen is also named, e.g. "bladder catheter"), and clinical body-site terms → Clinical;
   9. otherwise → Unclassified.

**Why two layers:** the hand-checked lookup is the most accurate for values I have actually seen, and the keyword rules keep the pipeline from breaking when NCBI adds new strings.

Two conventions that other papers sometimes make differently are exposed as switches at the top of the rules section of the module: `HOSPITAL_ENVIRONMENT_AS` (hospital surfaces/sinks, default `"Environmental"`) and `ANIMAL_DERIVED_FOOD_AS` (retail meat/milk/eggs, default `"Animal"`).

The next cell just peeks into the module: the size of each curated list, and how `assign_group()` handles a few strings (the first three are in the lists, the rest fall through to the keyword rules).

In [ ]:
from kp_isolation_groups import (
    CLINICAL_SOURCES, ENVIRONMENTAL_SOURCES, ANIMAL_SOURCES, UNCLASSIFIED_SOURCES,
    EXACT_OVERRIDES, assign_group,
)

for name, lst in [("Clinical", CLINICAL_SOURCES), ("Environmental", ENVIRONMENTAL_SOURCES),
                  ("Animal", ANIMAL_SOURCES), ("Unclassified", UNCLASSIFIED_SOURCES)]:
    print(f"{name:<14}{len(lst):>6,} curated strings")
print(f"{'overrides':<14}{len(EXACT_OVERRIDES):>6,} exact overrides in the rule engine")

examples = ["blood", "cow milk", "hospital effluent",                 # found in the curated lists
            "chicken cloacal swab", "icu bed rail",                   # not in the lists -> keyword rules
            "mutant strain induced by colistin", "brand new weird thing"]
for e in examples:
    print(f"{e:<40} -> {assign_group(e)}")

The wrapper below applies `assign_group` to every row with `Series.map` and prints the group counts. The import sits inside the function so the big module is only loaded when this step runs.

In [ ]:
def add_isolation_group(df):
    """
    There are ~2,900 different Isolation_source strings so i grouped them into
    4 categories. The big lists + keyword rules live in kp_isolation_groups.py
    (keeping them in their own file because they are very long).
    """
    from kp_isolation_groups import assign_group

    df["Isolation_Group"] = df["Isolation_source"].map(assign_group)
    print(df["Isolation_Group"].value_counts())
    return df

In [ ]:
df = add_isolation_group(df)

### Step 9: save the cleaned table

`index=False` keeps the row index out of the file.

In [ ]:
df.to_csv(CLEAN_FILE, index=False)
print(f"saved {CLEAN_FILE}  ({df.shape[0]:,} rows x {df.shape[1]} cols)")

# Part B: Sampling

### Step 10: create `Continent` from `Location` (Module 2)

`Location` looks like `"usa"`, `"china: hangzhou"`, `"usa: san antonio, tx"` or `"united kingdom (england, wales & n. ireland)"`, so the country is the text before the first colon. `assign_continent()` (in `kp_continent_groups.py`) works like this:

1. `extract_country()` splits on the first `":"` and checks whether that head is a key of `COUNTRY_TO_CONTINENT`, a dictionary of 175 entries grouped by the UN geoscheme.
2. If there is no colon (or the head is not a key), it falls back to a *starts-with* match against the keys, **sorted longest first**, so a multi-word name like `"south africa"` is tested before any shorter key. It only matches when the key is followed by a space or `(`, so `"niger"` can never match `"nigeria"`.
3. The country is looked up in the dictionary. Anything it cannot place becomes `"Unknown"` (55 rows in my run, all `"missing: data agreement established pre-2023"`).

Special cases (documented at the top of the module): Gaza Strip and West Bank go to Asia (Middle East), Hong Kong and Taiwan are kept as written in the source data and mapped to Asia, and Antarctica stays its own group.

In [ ]:
from kp_continent_groups import COUNTRY_TO_CONTINENT, extract_country, assign_continent

print(len(COUNTRY_TO_CONTINENT), "entries in the country -> continent table")

for loc in ["usa", "china: hangzhou", "usa: san antonio, tx",
            "united kingdom (england, wales & n. ireland)",
            "missing: data agreement established pre-2023"]:
    print(f"{loc:<48} -> country={extract_country(loc)!r:<48} continent={assign_continent(loc)}")

In [ ]:
def add_continent(df):
    """
    Location looks like "usa", "china: hangzhou", "usa: san antonio, tx", so the
    country is the part before the ":". kp_continent_groups.py looks that country
    up in a country -> continent table (also its own file because it is long).
    Anything it can't place gets "Unknown" (55 rows in my run, all
    "missing: data agreement established pre-2023").
    """
    from kp_continent_groups import assign_continent

    df["Continent"] = df["Location"].map(assign_continent)
    print(df["Continent"].value_counts(dropna=False))
    return df

In [ ]:
df = add_continent(df)

### Step 11: keep only recent genomes

A boolean filter on `Collection_year`. `.copy()` makes an independent DataFrame, which avoids pandas' `SettingWithCopyWarning` if I modify it later.

In [ ]:
def keep_recent_years(df):
    recent = df[df["Collection_year"] >= SAMPLE_MIN_YEAR].copy()
    log_rows(recent, f"genomes collected in {SAMPLE_MIN_YEAR} or later")
    return recent

In [ ]:
df = keep_recent_years(df)

#### Optional: remove duplicate genomes (`GCF_` vs `GCA_`)

The same genome can be listed twice, once as a RefSeq accession (`GCF_`) and once as a GenBank accession (`GCA_`). Because the `Assembly` column is unique it does not *look* like a duplicate. How the function removes them:

1. Build a key from the accession without its prefix (regex `^gc[af]_` removed), so `gcf_000709245.2` and `gca_000709245.2` share the key `000709245.2`.
2. Sort with a boolean key (`startswith("gca_")`: `False` sorts before `True`), so `GCF_` rows come first. `kind="stable"` keeps the original order among ties.
3. `drop_duplicates(subset="_key", keep="first")` keeps the first row per key, i.e. the `GCF_` copy when both exist.
4. `sort_index()` restores the original row order.

It is off by default (`DROP_GCA_DUPLICATES = False`) because switching it on changes which 300 genomes get picked.

In [ ]:
def drop_gca_duplicates(df):
    """
    gcf_000709245.2 and gca_000709245.2 are the same genome. i make a key from
    the number part, sort so the GCF_ one comes first, and keep only the first.
    """
    key = df["Assembly"].str.replace(r"^gc[af]_", "", regex=True)
    df = df.assign(_key=key)
    df = df.sort_values("Assembly", key=lambda s: s.str.startswith("gca_"), kind="stable")
    df = df.drop_duplicates(subset="_key", keep="first").drop(columns="_key")
    df = df.sort_index()
    log_rows(df, "after removing GCA_ copies of GCF_ genomes")
    return df

In [ ]:
if DROP_GCA_DUPLICATES:
    df = drop_gca_duplicates(df)

### Step 12: is there enough data in each box?

I build a contingency table (isolation group × continent) *before* sampling, because `.sample(n=20)` raises an error on any group with fewer than 20 rows (this is exactly what happens for Animal + Oceania).

- `groupby([...]).size()` counts the rows per combination.
- `unstack(fill_value=0)` pivots the continent level into columns and fills empty combinations with 0.
- `reindex(...)` forces a fixed row/column order and makes sure missing combinations still appear as 0.

Only the boxes I actually sample from (`SAMPLE_CONTINENTS`) have to reach `N_PER_BOX`, so Oceania is shown in the table but not checked. If a required box is too small the function raises a `ValueError` that lists it.

In [ ]:
def count_per_box(df):
    """
    table of how many genomes we have for each group x continent combo.
    i look at this BEFORE sampling, otherwise .sample(n=20) crashes on a box
    that has fewer than 20 (that is what happens with animal + oceania).
    """
    counts = (
        df[df["Isolation_Group"].isin(SAMPLE_GROUPS) & df["Continent"].isin(ALL_CONTINENTS)]
        .groupby(["Isolation_Group", "Continent"])
        .size()
        .unstack(fill_value=0)
        .reindex(index=SAMPLE_GROUPS, columns=ALL_CONTINENTS, fill_value=0)
    )
    print("genomes available in each box:")
    print(counts)

    # only the boxes i actually sample from have to be big enough
    needed = counts[SAMPLE_CONTINENTS]
    too_small = needed[needed < N_PER_BOX].stack()
    too_small = too_small[too_small < N_PER_BOX]
    if len(too_small) > 0:
        raise ValueError(f"not enough genomes for {N_PER_BOX} per box:\n{too_small}")
    return counts

In [ ]:
counts = count_per_box(df)

### Step 13: stratified random sample

Clinical isolates are ~90% of the data, so a simple random sample would be almost all clinical. Instead I draw the **same number (20) from every stratum** (equal allocation, not proportional), so the three sources are equally represented.

`groupby(["Isolation_Group", "Continent"]).sample(n=N_PER_BOX, random_state=RANDOM_SEED)` samples inside each group. `group_keys=False` keeps a flat index, and the fixed `random_state` means the same 300 genomes are drawn every run.

In [ ]:
def draw_sample(df):
    """
    20 random genomes from every (Isolation_Group, Continent) box so the three
    sources are equally represented, not dominated by clinical isolates
    (clinical is ~90% of the data otherwise).
    """
    sample = (
        df[df["Isolation_Group"].isin(SAMPLE_GROUPS) & df["Continent"].isin(SAMPLE_CONTINENTS)]
        .groupby(["Isolation_Group", "Continent"], group_keys=False)
        .sample(n=N_PER_BOX, random_state=RANDOM_SEED)
        .reset_index(drop=True)
    )

    print("sample size:", len(sample))
    print(sample.groupby(["Isolation_Group", "Continent"]).size().unstack(fill_value=0))
    return sample

In [ ]:
sample = draw_sample(df)

### Step 14: save the sample and the accession lists

The cleaning step lowercased everything, including `Assembly`, but NCBI accessions (`GCF_` / `GCA_`) are uppercase, so I apply `.str.upper()` before writing. One text file per isolation group, one accession per line, ready for downloading the assemblies.

In [ ]:
sample.to_csv(SAMPLE_FILE, index=False)
print(f"saved {SAMPLE_FILE}  ({sample.shape[0]} rows x {sample.shape[1]} cols)")

In [ ]:
def save_accession_lists(sample):
    """
    one text file per isolation group (clinical_assembly.txt etc.), one accession
    per line. i uppercase them here because NCBI wants GCF_/GCA_ in capitals
    and my cleaning step lowercased everything.
    """
    for group in SAMPLE_GROUPS:
        accessions = sample.loc[sample["Isolation_Group"] == group, "Assembly"].str.upper()
        filename = f"{group.lower()}_assembly.txt"
        with open(filename, "w", encoding="utf-8") as f:
            f.writelines(f"{acc}\n" for acc in accessions)
        print(f"saved {filename}  ({len(accessions)} accessions)")

In [ ]:
save_accession_lists(sample)

## Things I noticed while working

1. **`YEAR_MAX`.** The data has genomes from 2026 (about 1,478 rows in my run). With `YEAR_MAX = 2025` they get dropped. If 2026 counts for the project, change `YEAR_MAX` to 2026 and rerun.
2. **Duplicate genomes.** The same genome can appear twice, as a RefSeq accession (`GCF_`) and as a GenBank accession (`GCA_`), e.g. `gcf_000709245.2` and `gca_000709245.2`. The `Assembly` column is unique, so it does not look like a duplicate, but it is. This can put the same genome in the sample twice. Set `DROP_GCA_DUPLICATES = True` to remove them before sampling (it changes which 300 genomes are picked).
3. **Lowercase accessions.** Everything is lowercased, including `Assembly`. NCBI accessions are uppercase, so the accession lists are written with `.upper()`. The CSV files still have them in lowercase.
4. **`Organism`.** Some rows still have strain names stuck on the end (e.g. `"klebsiella pneumoniae uhkpc05"`), 171 different values in total.
5. **Oceania.** It has no animal genomes from 2018 onward, so it is left out of the sample (it only shows up in the counts table).

## Why it is built this way

- **Hidden missing values.** Text like `"missing"` looks like real data to pandas, so a plain `dropna()` silently keeps those rows. Converting such words to real `NaN` first and then auditing makes the cleaning *verifiable* instead of assumed.
- **Grouping before analysis.** Resistance patterns are compared between human, animal and environmental sources, and between regions. That only works once thousands of free-text labels are turned into a few consistent categories.
- **Stratified sampling.** Clinical isolates make up most of the dataset, so a simple random sample would be almost all clinical. Sampling the same number from each source × continent box keeps the groups equally represented.
- **Counting before sampling.** Oceania has no animal genomes after 2018, so asking for 20 per box there would crash. Checking the counts first showed this, and Oceania was left out of the sample on purpose.
- **Uppercase accessions.** Cleaning lowercases everything, but NCBI accession numbers are uppercase (`GCF_` / `GCA_`), so the accession lists are converted back before downloading.